In [1]:
import pandas as pd

df = pd.read_excel("AllProducts with ASIN_NE.xlsx")
df

,NOVICA SKU,PID,Att,ASIN,URL,Auction Title,SKU Type,Total Quantity,Quantity Update Type,Quantity,...,Picture URLs,Classification,Sales Last 365 Days,brand_text_length,brand_keywords,brand_sentiment_words,description_length,top_keywords,unique_keyword_count,sentiment_keywords
0,988,988,NaN,B0010XKQWS,https://www.amazon.com/dp/B0010XKQWS,"Wool rug, 'Floral Bud' (6x8)",Basic Item,1.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Area Rugs,0,0,0,0,0,0,0,0
1,990,990,NaN,B001DKDBBG,https://www.amazon.com/dp/B001DKDBBG,"Wool runner, 'Spring Buds' (2x5.5)",Basic Item,1.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Area Rugs,0,0,0,0,0,0,0,0
2,1219,1219,NaN,B000160Y54,https://www.amazon.com/dp/B000160Y54,"Cedar and leather trays, 'Tumi' (set of 3)",Basic Item,3.0,NaN,NaN,...,ITEMIMAGEURL1=https://images1.novica.net/pictu...,Tableware & Entertaining,0,0,0,0,0,0,0,0
3,1227,1227,NaN,B0009G0WBC,https://www.amazon.com/dp/B0009G0WBC,"Cedar and leather magazine rack, 'Nobility'",Basic Item,0.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Furniture,1,0,0,0,0,"novica, artisan, handmade, cedar, leather, woo...",42,"artisan, handmade, quality, unique, traditiona..."
4,3285,3285,NaN,B00011V0RU,https://www.amazon.com/dp/B00011V0RU,"Wood mask, 'Beautiful Sita'",Basic Item,1.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Masks,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42259,96582a63760,96582,63760.0,B081FT62VZ,https://www.amazon.com/dp/B081FT62VZ,"Peridot cocktail ring, 'Lime Halo'",Child,0.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Jewelry,0,0,0,0,0,0,0,0
42260,96582a63761,96582,63761.0,B081G75X96,https://www.amazon.com/dp/B081G75X96,"Peridot cocktail ring, 'Lime Halo'",Child,0.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Jewelry,0,0,0,0,0,0,0,0
42261,96584a52284,96584,52284.0,B09BNZSDNX,https://www.amazon.com/dp/B09BNZSDNX,"Blue topaz cocktail ring, 'Wink'",Child,0.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Jewelry,0,0,0,0,0,0,0,0
42262,96584a52285,96584,52285.0,B09BP47B5S,https://www.amazon.com/dp/B09BP47B5S,"Blue topaz cocktail ring, 'Wink'",Child,0.0,NaN,NaN,...,ITEMIMAGEURL1=http://images1.novica.net/pictur...,Jewelry,0,0,0,0,0,0,0,0


In [2]:
# Step 1: Ensure keyword fields are strings
df["top_keywords"] = df["top_keywords"].fillna("").astype(str)
df["unique_keyword_count"] = df["unique_keyword_count"].fillna("").astype(str)

# Step 2: Combine all keyword sources into one list per SKU
df["combined_keywords"] = df["top_keywords"] + "," + df["unique_keyword_count"]

# Step 3: Convert comma-separated keywords into list
df["combined_keywords"] = df["combined_keywords"].apply(lambda x: [k.strip().lower() for k in x.split(",") if k])

In [3]:
from collections import Counter

keyword_counts = Counter()

for kw_list in df["combined_keywords"]:
    keyword_counts.update(kw_list)

keyword_df = pd.DataFrame.from_dict(keyword_counts, orient="index", columns=["frequency"])
keyword_df.head()

,frequency
0,71584
novica,6469
artisan,5706
handmade,5773
cedar,14


In [4]:
import numpy as np

sales = df["Sales Last 365 Days"]

keyword_df["correlation"] = 0.0

for keyword in keyword_df.index:
    df[keyword] = df["combined_keywords"].apply(lambda x: 1 if keyword in x else 0)
    keyword_df.loc[keyword, "correlation"] = np.corrcoef(df[keyword], sales)[0,1]

C:\Users\32lio\AppData\Local\Temp\ipykernel_24800\1095963585.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[keyword] = df["combined_keywords"].apply(lambda x: 1 if keyword in x else 0)
C:\Users\32lio\AppData\Local\Temp\ipykernel_24800\1095963585.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[keyword] = df["combined_keywords"].apply(lambda x: 1 if keyword in x else 0)
C:\Users\32lio\AppData\Local\Temp\ipykernel_24800\1095963585.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result o

In [5]:
import numpy as np

# If you have a numeric sentiment column per SKU (from your screenshot)
if "brand_sentiment" in df.columns:
    # Make sure it's numeric
    df["brand_sentiment"] = pd.to_numeric(df["brand_sentiment"], errors="coerce").fillna(0)

    # Collect sentiment values per keyword
    keyword_sentiment = {}

    for i, row in df.iterrows():
        s = row["brand_sentiment"]
        for kw in row["combined_keywords"]:
            keyword_sentiment.setdefault(kw, []).append(s)

    # Average sentiment for each keyword
    keyword_df["sentiment"] = keyword_df.index.map(
        lambda k: np.mean(keyword_sentiment.get(k, [0]))
    )
else:
    # Fallback: no sentiment column → set to 0
    keyword_df["sentiment"] = 0.0

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor

# Join keywords back into a space-separated string per SKU
df["text_keywords"] = df["combined_keywords"].apply(lambda ks: " ".join(ks))

# TF-IDF representation of keywords
tfidf = TfidfVectorizer(max_features=3000)  # you can change the limit
X = tfidf.fit_transform(df["text_keywords"])

y = df["Sales Last 365 Days"].values

# Random Forest regressor
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X, y)

# Map importance back to individual terms
feature_names = tfidf.get_feature_names_out()
importances = rf.feature_importances_

importance_map = dict(zip(feature_names, importances))

# Add importance to keyword_df (0 if keyword not in TF-IDF vocab)
keyword_df["importance"] = keyword_df.index.map(
    lambda k: importance_map.get(k, 0.0)
)

C:\Users\32lio\AppData\Local\Temp\ipykernel_24800\4198788448.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["text_keywords"] = df["combined_keywords"].apply(lambda ks: " ".join(ks))


In [7]:
from sklearn.preprocessing import MinMaxScaler

# Use absolute correlation strength
keyword_df["correlation"] = keyword_df["correlation"].abs()

# Replace any NaNs with 0 before scaling
for col in ["frequency", "correlation", "sentiment", "importance"]:
    keyword_df[col] = keyword_df[col].fillna(0)

scaler = MinMaxScaler()

keyword_df[["freq_scaled", "corr_scaled", "sent_scaled", "imp_scaled"]] = scaler.fit_transform(
    keyword_df[["frequency", "correlation", "sentiment", "importance"]]
)

In [8]:
# Simple average of all four scaled components
keyword_df["impact_score"] = (
    keyword_df["freq_scaled"]
    + keyword_df["corr_scaled"]
    + keyword_df["sent_scaled"]
    + keyword_df["imp_scaled"]
) / 4.0

# Sort by impact score (highest = best keyword)
keyword_df_sorted = keyword_df.sort_values("impact_score", ascending=False)

# Look at top 30 keywords
top_keywords = keyword_df_sorted.head(30)
top_keywords

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score
novica,6469,0.943131,0.0,0.902022,0.090357,0.999734,0.0,1.000000e+00,0.522523
0,71584,0.943380,0.0,0.000000,1.000000,1.000000,0.0,1.557472e-18,0.500000
handmade,5773,0.881631,0.0,0.000173,0.080634,0.934121,0.0,1.922733e-04,0.253737
artisan,5706,0.876795,0.0,0.000204,0.079698,0.928961,0.0,2.264690e-04,0.252221
silver,3862,0.694491,0.0,0.000079,0.053937,0.734460,0.0,8.742162e-05,0.197121
sterling,3769,0.685174,0.0,0.000061,0.052638,0.724520,0.0,6.733240e-05,0.194306
artisans,3400,0.662350,0.0,0.000131,0.047483,0.700169,0.0,1.456014e-04,0.186949
world,2599,0.566620,0.0,0.000060,0.036294,0.598034,0.0,6.686378e-05,0.158599
works,2388,0.542857,0.0,0.000072,0.033346,0.572681,0.0,8.018442e-05,0.151527
together,1965,0.486841,0.0,0.000023,0.027437,0.512918,0.0,2.557552e-05,0.135095


In [9]:
# remove noise keywords
noise = {
    "novica", "0", "and", "for", "the", "with", "of", 
    "world", "works", "together", "around", "indonesia","arriola","produce","time","spread",'customers',"keepsake"
}

keyword_df = keyword_df[~keyword_df.index.isin(noise)]

# remove single-character or numeric only words
keyword_df = keyword_df[keyword_df.index.str.isalpha()]

# optional: remove extremely high frequency generic words
keyword_df = keyword_df[keyword_df["frequency"] < 10000]
keyword_df.head()

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score
artisan,5706,0.876795,0.0,2.042801e-04,0.079698,0.928961,0.0,2.264690e-04,0.252221
handmade,5773,0.881631,0.0,1.734348e-04,0.080634,0.934121,0.0,1.922733e-04,0.253737
cedar,14,0.039537,0.0,9.073108e-20,0.000182,0.035687,0.0,1.658058e-18,0.008967
leather,143,0.130748,0.0,1.154695e-06,0.001984,0.133001,0.0,1.280118e-06,0.033746
wood,834,0.320424,0.0,6.348038e-05,0.011637,0.335366,0.0,7.037562e-05,0.086768


In [10]:
from sklearn.preprocessing import MinMaxScaler

# Fill any remaining NaNs
for col in ["frequency", "correlation", "sentiment", "importance"]:
    keyword_df[col] = keyword_df[col].fillna(0)

# use absolute correlation
keyword_df["correlation"] = keyword_df["correlation"].abs()

scaler = MinMaxScaler()

keyword_df[["freq_scaled", "corr_scaled", "sent_scaled", "imp_scaled"]] = scaler.fit_transform(
    keyword_df[["frequency", "correlation", "sentiment", "importance"]]
)

In [11]:
keyword_df["impact_score"] = (
    keyword_df["freq_scaled"]
    + keyword_df["corr_scaled"]
    + keyword_df["sent_scaled"]
    + keyword_df["imp_scaled"]
) / 4.0

# Sort the keywords by score
keyword_df_sorted = keyword_df.sort_values("impact_score", ascending=False)

keyword_df_sorted.head(30)

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score
handmade,5773,0.881631,0.0,0.000173,1.000000,1.000000,0.0,0.013847,0.503462
artisan,5706,0.876795,0.0,0.000204,0.988392,0.994476,0.0,0.016309,0.499794
silver,3862,0.694491,0.0,0.000079,0.668919,0.786258,0.0,0.006296,0.365368
sterling,3769,0.685174,0.0,0.000061,0.652807,0.775617,0.0,0.004849,0.358318
artisans,3400,0.662350,0.0,0.000131,0.588877,0.749549,0.0,0.010485,0.337228
bonds,3,0.083362,0.0,0.012525,0.000347,0.088259,0.0,1.000000,0.272151
talented,1865,0.473489,0.0,0.000046,0.322938,0.533841,0.0,0.003658,0.215109
birthstone,1857,0.470374,0.0,0.000044,0.321552,0.530284,0.0,0.003518,0.213838
artists,1835,0.469601,0.0,0.000035,0.317741,0.529401,0.0,0.002783,0.212481
designers,1713,0.453202,0.0,0.000025,0.296604,0.510671,0.0,0.001994,0.202317


In [12]:
top_20 = keyword_df_sorted.head(20)
top_20

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score
handmade,5773,0.881631,0.0,0.000173,1.000000,1.000000,0.0,0.013847,0.503462
artisan,5706,0.876795,0.0,0.000204,0.988392,0.994476,0.0,0.016309,0.499794
silver,3862,0.694491,0.0,0.000079,0.668919,0.786258,0.0,0.006296,0.365368
sterling,3769,0.685174,0.0,0.000061,0.652807,0.775617,0.0,0.004849,0.358318
artisans,3400,0.662350,0.0,0.000131,0.588877,0.749549,0.0,0.010485,0.337228
bonds,3,0.083362,0.0,0.012525,0.000347,0.088259,0.0,1.000000,0.272151
talented,1865,0.473489,0.0,0.000046,0.322938,0.533841,0.0,0.003658,0.215109
birthstone,1857,0.470374,0.0,0.000044,0.321552,0.530284,0.0,0.003518,0.213838
artists,1835,0.469601,0.0,0.000035,0.317741,0.529401,0.0,0.002783,0.212481
designers,1713,0.453202,0.0,0.000025,0.296604,0.510671,0.0,0.001994,0.202317


In [13]:
keyword_df["impact_score"] = (
    0.4 * keyword_df["corr_scaled"] +
    0.3 * keyword_df["freq_scaled"] +
    0.2 * keyword_df["imp_scaled"] +
    0.1 * keyword_df["sent_scaled"]
)

In [14]:
keyword_df["impact_score_100"] = keyword_df["impact_score"] * 100

In [15]:
keyword_df["impact_score_simple"] = (
    keyword_df["freq_scaled"] + keyword_df["corr_scaled"]
) / 2

In [16]:
keyword_df_sorted.head(20)

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score
handmade,5773,0.881631,0.0,0.000173,1.000000,1.000000,0.0,0.013847,0.503462
artisan,5706,0.876795,0.0,0.000204,0.988392,0.994476,0.0,0.016309,0.499794
silver,3862,0.694491,0.0,0.000079,0.668919,0.786258,0.0,0.006296,0.365368
sterling,3769,0.685174,0.0,0.000061,0.652807,0.775617,0.0,0.004849,0.358318
artisans,3400,0.662350,0.0,0.000131,0.588877,0.749549,0.0,0.010485,0.337228
bonds,3,0.083362,0.0,0.012525,0.000347,0.088259,0.0,1.000000,0.272151
talented,1865,0.473489,0.0,0.000046,0.322938,0.533841,0.0,0.003658,0.215109
birthstone,1857,0.470374,0.0,0.000044,0.321552,0.530284,0.0,0.003518,0.213838
artists,1835,0.469601,0.0,0.000035,0.317741,0.529401,0.0,0.002783,0.212481
designers,1713,0.453202,0.0,0.000025,0.296604,0.510671,0.0,0.001994,0.202317


In [17]:
# After calculating impact_score
keyword_df_sorted = keyword_df.sort_values("impact_score", ascending=False)

# Show top 20 keywords
keyword_df_sorted.head(20)

,frequency,correlation,sentiment,importance,freq_scaled,corr_scaled,sent_scaled,imp_scaled,impact_score,impact_score_100,impact_score_simple
handmade,5773,0.881631,0.0,0.000173,1.000000,1.000000,0.0,0.013847,0.702769,70.276931,1.000000
artisan,5706,0.876795,0.0,0.000204,0.988392,0.994476,0.0,0.016309,0.697570,69.756989,0.991434
silver,3862,0.694491,0.0,0.000079,0.668919,0.786258,0.0,0.006296,0.516438,51.643803,0.727588
sterling,3769,0.685174,0.0,0.000061,0.652807,0.775617,0.0,0.004849,0.507058,50.705849,0.714212
artisans,3400,0.662350,0.0,0.000131,0.588877,0.749549,0.0,0.010485,0.478580,47.857975,0.669213
talented,1865,0.473489,0.0,0.000046,0.322938,0.533841,0.0,0.003658,0.311149,31.114947,0.428390
birthstone,1857,0.470374,0.0,0.000044,0.321552,0.530284,0.0,0.003518,0.309283,30.928280,0.425918
artists,1835,0.469601,0.0,0.000035,0.317741,0.529401,0.0,0.002783,0.307639,30.763920,0.423571
designers,1713,0.453202,0.0,0.000025,0.296604,0.510671,0.0,0.001994,0.293648,29.364842,0.403638
ring,1619,0.435071,0.0,0.000015,0.280319,0.489962,0.0,0.001221,0.280325,28.032480,0.385141


In [18]:
output_df = keyword_df_sorted[["impact_score", "impact_score_100"]]

output_df.head(20)

,impact_score,impact_score_100
handmade,0.702769,70.276931
artisan,0.697570,69.756989
silver,0.516438,51.643803
sterling,0.507058,50.705849
artisans,0.478580,47.857975
talented,0.311149,31.114947
birthstone,0.309283,30.928280
artists,0.307639,30.763920
designers,0.293648,29.364842
ring,0.280325,28.032480


In [19]:
# === GAP ANALYSIS FOR TOP N KEYWORDS ===

TOP_N = 100   # change to 30, 50, etc. if you like

# 1. Take the top N keywords by impact_score
top_keywords = keyword_df_sorted.head(TOP_N).copy()

# 2. Benchmark = mean impact score of these top N keywords
benchmark_score = top_keywords["impact_score"].mean()

# 3. Correct GAP: how much the keyword is ABOVE/BELOW the benchmark
#    Positive gap  -> performing better than benchmark
#    Negative gap  -> performing worse than benchmark
top_keywords["gap_vs_benchmark"] = top_keywords["impact_score"] - benchmark_score

# 4. Priority logic
#    High Priority   = high frequency + clearly above benchmark
#    Medium Priority = around benchmark
#    Low Priority    = clearly below benchmark
def priority_level(row):
    if row["gap_vs_benchmark"] > 0.02 and row["freq_scaled"] > 0.5:
        return "High Priority"
    elif row["gap_vs_benchmark"] > -0.02:
        return "Medium Priority"
    else:
        return "Low Priority"

top_keywords["priority"] = top_keywords.apply(priority_level, axis=1)

# 5. Final GAP table
gap_analysis_df = top_keywords[
    [
        "impact_score",
        "freq_scaled",
        "corr_scaled",
        "imp_scaled",
        "sent_scaled",
        "gap_vs_benchmark",
        "priority",
    ]
]

gap_analysis_df.head(TOP_N)

,impact_score,freq_scaled,corr_scaled,imp_scaled,sent_scaled,gap_vs_benchmark,priority
handmade,0.702769,1.000000,1.000000,0.013847,0.0,0.529643,High Priority
artisan,0.697570,0.988392,0.994476,0.016309,0.0,0.524444,High Priority
silver,0.516438,0.668919,0.786258,0.006296,0.0,0.343312,High Priority
sterling,0.507058,0.652807,0.775617,0.004849,0.0,0.333932,High Priority
artisans,0.478580,0.588877,0.749549,0.010485,0.0,0.305454,High Priority
...,...,...,...,...,...,...,...
painted,0.093691,0.039155,0.202941,0.003841,0.0,-0.079435,Low Priority
diam,0.092455,0.045218,0.196949,0.000551,0.0,-0.080671,Low Priority
card,0.090862,0.039328,0.197586,0.000146,0.0,-0.082264,Low Priority
guatemala,0.090444,0.041753,0.193286,0.003019,0.0,-0.082682,Low Priority


In [20]:
import pandas as pd

# --- 1. Load BOTTOM 2500 sheet ---
df_bottom = pd.read_excel("TOP 2500 final.xlsx", sheet_name="BOTTOM 2500")

# --- 1a. Detect the keyword column dynamically ---
# We look for any column that contains "keyword" in its name.
keyword_cols = [c for c in df_bottom.columns if "keyword" in c.lower()]

if not keyword_cols:
    raise KeyError("No keyword-like column found. Available columns are: "
                   f"{list(df_bottom.columns)}")

keyword_col = keyword_cols[0]  # use the first match (e.g., 'top_keywords')

print(f"Using keyword column: {keyword_col}")

# --- 2. Build a smaller dataframe with SKU, Classification, and keyword column ---
# If your sheet does not have 'Classification', you can remove it from the list below.
cols_to_use = ["SKU", keyword_col]
if "Classification" in df_bottom.columns:
    cols_to_use.insert(1, "Classification")  # ['SKU', 'Classification', keyword_col]

df_bottom_small = df_bottom[cols_to_use].copy()

# --- 3. Explode comma-separated keywords into one row per SKU–keyword ---
df_bottom_small["keyword_list"] = (
    df_bottom_small[keyword_col]
    .fillna("")
    .apply(lambda x: [k.strip() for k in str(x).split(",") if k.strip() != ""])
)

df_exploded = df_bottom_small.explode("keyword_list").rename(
    columns={"keyword_list": "keyword"}
)

# --- 4. Merge with global keyword metrics (keyword_df_sorted) ---
# Make sure keyword_df_sorted exists and has these columns
keyword_metrics = keyword_df_sorted[
    ["impact_score", "freq_scaled", "corr_scaled",
     "imp_scaled", "sent_scaled"]
].copy()

merged = df_exploded.merge(keyword_metrics, on="keyword", how="left")

# Drop keywords we don't have metrics for
merged = merged.dropna(subset=["impact_score"])

# --- 5. Global benchmark (same for all SKUs) ---
global_benchmark = merged["impact_score"].mean()

# Correct GAP: how far above/below the benchmark each keyword is
merged["gap_vs_benchmark"] = merged["impact_score"] - global_benchmark

# --- 6. Priority logic (per SKU–keyword, using global benchmark) ---
def priority_level_global(row):
    if row["gap_vs_benchmark"] > 0.02 and row["freq_scaled"] > 0.5:
        return "High Priority"
    elif row["gap_vs_benchmark"] > -0.02:
        return "Medium Priority"
    else:
        return "Low Priority"

merged["priority"] = merged.apply(priority_level_global, axis=1)

# --- 7. Final gap analysis table (GLOBAL benchmark) ---
cols_out = ["SKU", "keyword", "impact_score", "freq_scaled", "corr_scaled",
            "imp_scaled", "sent_scaled", "gap_vs_benchmark", "priority"]

if "Classification" in merged.columns:
    cols_out.insert(1, "Classification")  # add Classification after SKU if present

gap_analysis_global = merged[cols_out]

gap_analysis_global.head(20)

Using keyword column: brand_keywords


KeyError: 'keyword'